# Sesión 6 · Aprendizaje no supervisado y detección de anomalías
**IA aplicada a la ciberseguridad · Máster en Ciberseguridad**

| Parte | Qué haces |
|---|---|
| A | k-means a mano y con scikit-learn; elegir k |
| B | DBSCAN y la curva de distancias |
| C | PCA como detector: error de reconstrucción |
| D | Isolation Forest, LOF y One-Class SVM |
| E | Anomalías en un auth.log sin etiquetas |
| F | z robusto sobre una serie de DNS |
| G | Presupuesto de alertas y precision@k |

**Regla de la sesión:** los modelos se entrenan **sin etiquetas** (o solo con tráfico normal). Las etiquetas se usan únicamente para **medir**.

> **Dos rutas en este cuaderno.** La **ruta base** ejecuta y observa: cada parte explica qué deberías ver. Los bloques **Reto avanzado** son opcionales y piden método.
>
> Ruta base: partes A, C, D, F y G. Ruta avanzada: retos, B y E.
>
> Ejecuta las celdas **en orden** (`Mayús + Intro`). En Colab, si se desconecta el entorno, vuelve a ejecutar la preparación.

## Preparación

In [ ]:
# Preparación del entorno: ejecuta esta celda la primera (Colab, VS Code o Codespaces)
import sys, os, subprocess, importlib.util
EN_COLAB = "google.colab" in sys.modules
print("Entorno:", "Google Colab" if EN_COLAB else "Jupyter en VS Code / Codespaces")
paquetes = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib",
            "seaborn": "seaborn", "joblib": "joblib", "xgboost": "xgboost", "lightgbm": "lightgbm"}
faltan = [pip for mod, pip in paquetes.items() if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])

# Funciones del curso en ia_utils.py (no modifica tu utils.py)
IA_UTILS = "\"\"\"Funciones del curso IA aplicada a la ciberseguridad (sesiones 3 a 6).\"\"\"\nimport os, re, urllib.request\nimport numpy as np\nimport pandas as pd\n\nNSL_URL = \"https://raw.githubusercontent.com/defcom17/NSL_KDD/master/\"\nNSL_COLS = [\"duration\",\"protocol_type\",\"service\",\"flag\",\"src_bytes\",\"dst_bytes\",\"land\",\n  \"wrong_fragment\",\"urgent\",\"hot\",\"num_failed_logins\",\"logged_in\",\"num_compromised\",\n  \"root_shell\",\"su_attempted\",\"num_root\",\"num_file_creations\",\"num_shells\",\n  \"num_access_files\",\"num_outbound_cmds\",\"is_host_login\",\"is_guest_login\",\"count\",\n  \"srv_count\",\"serror_rate\",\"srv_serror_rate\",\"rerror_rate\",\"srv_rerror_rate\",\n  \"same_srv_rate\",\"diff_srv_rate\",\"srv_diff_host_rate\",\"dst_host_count\",\n  \"dst_host_srv_count\",\"dst_host_same_srv_rate\",\"dst_host_diff_srv_rate\",\n  \"dst_host_same_src_port_rate\",\"dst_host_srv_diff_host_rate\",\"dst_host_serror_rate\",\n  \"dst_host_srv_serror_rate\",\"dst_host_rerror_rate\",\"dst_host_srv_rerror_rate\",\n  \"label\",\"difficulty\"]\n\n\ndef load_nsl_kdd(split=\"train\", base=\"data\"):\n    \"\"\"Carga NSL-KDD y añade la columna is_attack. Descarga los ficheros si faltan.\"\"\"\n    name = \"KDDTrain+.txt\" if split == \"train\" else \"KDDTest+.txt\"\n    path = os.path.join(base, name)\n    if not os.path.exists(path):\n        os.makedirs(base, exist_ok=True)\n        urllib.request.urlretrieve(NSL_URL + name.replace(\"+\", \"%2B\"), path)\n    df = pd.read_csv(path, header=None, names=NSL_COLS)\n    df[\"is_attack\"] = (df[\"label\"] != \"normal\").astype(int)\n    return df\n\n\nATTACK_MAP = {\"normal\": \"normal\",\n  **{a: \"dos\" for a in [\"back\", \"land\", \"neptune\", \"pod\", \"smurf\", \"teardrop\",\n                        \"apache2\", \"udpstorm\", \"processtable\", \"mailbomb\"]},\n  **{a: \"probe\" for a in [\"satan\", \"ipsweep\", \"nmap\", \"portsweep\", \"mscan\", \"saint\"]},\n  **{a: \"r2l\" for a in [\"guess_passwd\", \"ftp_write\", \"imap\", \"phf\", \"multihop\",\n                        \"warezmaster\", \"warezclient\", \"spy\", \"xlock\", \"xsnoop\",\n                        \"snmpguess\", \"snmpgetattack\", \"httptunnel\", \"sendmail\",\n                        \"named\", \"worm\"]},\n  **{a: \"u2r\" for a in [\"buffer_overflow\", \"loadmodule\", \"rootkit\", \"perl\",\n                        \"sqlattack\", \"xterm\", \"ps\"]}}\n\n\ndef add_category(df):\n    \"\"\"Añade la columna category: normal, dos, probe, r2l o u2r.\"\"\"\n    df[\"category\"] = df[\"label\"].map(ATTACK_MAP)\n    return df\n\n\ndef generar_flujos(n=60_000, frac_ataque=0.25, seed=0, cambio_herramienta=False):\n    \"\"\"Flujos de red sintéticos con columnas al estilo CICIDS2017, incluidos sus defectos\n    (espacios en los nombres, infinitos, nulos y duplicados). Etiquetas BENIGN / DDoS.\n    Con cambio_herramienta=True el ataque llega en dos episodios con herramientas distintas.\"\"\"\n    rng = np.random.default_rng(seed)\n    nb, na = int(n * (1 - frac_ataque)), int(n * frac_ataque)\n    t0 = pd.Timestamp(\"2017-07-07 09:00:00\")\n    # --- tráfico benigno\n    port_b = rng.choice([443, 80, 53, 22, 8080, 3389, 0], nb, p=[.40, .20, .20, .05, .05, .02, .08])\n    port_b = np.where(port_b == 0, rng.integers(1024, 65535, nb), port_b)\n    dur_b = np.where(rng.random(nb) < 0.02, 0, rng.lognormal(11, 2, nb)).astype(int)\n    fwd_b = rng.poisson(8, nb) + 1\n    bwd_b = rng.poisson(7, nb)\n    fwl_b = np.where(rng.random(nb) < 0.15, rng.uniform(0, 10, nb), rng.normal(400, 250, nb)).clip(0)\n    bwl_b = rng.normal(700, 400, nb).clip(0)\n    syn_b = (rng.random(nb) < 0.3).astype(int)\n    win_b = rng.choice([29200, 65535, 8192, 14600, 64240, -1], nb, p=[.25, .25, .15, .1, .15, .1])\n    ts_b = rng.uniform(0, 8 * 3600, nb)\n    # --- ataque\n    if cambio_herramienta:\n        epi = rng.random(na) < 0.5\n        ts_a = np.where(epi, rng.uniform(1 * 3600, 2 * 3600, na), rng.uniform(5 * 3600, 6 * 3600, na))\n        port_a = np.where(epi, 80, rng.choice([8080, 443], na))\n        win_a = np.where(epi, 8192, rng.choice([29200, 65535, 64240], na))\n        fwl_a = np.where(epi, rng.normal(8, 6, na), rng.normal(60, 40, na)).clip(0)\n    else:\n        ts_a = rng.uniform(6.9 * 3600, 7.3 * 3600, na)\n        port_a = np.where(rng.random(na) < 0.9, 80, 443)\n        win_a = np.where(rng.random(na) < 0.95, 8192, 29200)\n        fwl_a = rng.normal(8, 6, na).clip(0)\n    dur_a = rng.lognormal(13, 1.5, na).astype(int)\n    fwd_a = rng.poisson(4, na) + 1\n    bwd_a = rng.poisson(3, na)\n    bwl_a = rng.normal(1500, 600, na).clip(0)\n    syn_a = (rng.random(na) < 0.6).astype(int)\n\n    def bloque(port, dur, fwd, bwd, fwl, bwl, syn, win, ts, label):\n        tot_f, tot_b = fwd * fwl, bwd * bwl\n        seg = np.maximum(dur, 0) / 1e6\n        with np.errstate(divide=\"ignore\", invalid=\"ignore\"):\n            fbs = (tot_f + tot_b) / seg\n            fps = (fwd + bwd) / seg\n            iat = np.where(fwd + bwd > 1, dur / np.maximum(fwd + bwd - 1, 1), np.nan)\n        return pd.DataFrame({\n            \" Destination Port\": port, \" Flow Duration\": dur, \" Total Fwd Packets\": fwd,\n            \" Total Backward Packets\": bwd, \"Total Length of Fwd Packets\": tot_f.round(1),\n            \" Total Length of Bwd Packets\": tot_b.round(1), \" Fwd Packet Length Mean\": fwl.round(2),\n            \" Bwd Packet Length Mean\": bwl.round(2), \"Flow Bytes/s\": fbs, \" Flow Packets/s\": fps,\n            \" Flow IAT Mean\": iat, \" SYN Flag Count\": syn, \" ACK Flag Count\": (rng.random(len(port)) < 0.85).astype(int),\n            \"Init_Win_bytes_forward\": win, \" Avg Fwd Segment Size\": fwl.round(2),\n            \" Timestamp\": t0 + pd.to_timedelta(ts, unit=\"s\"), \" Label\": label})\n\n    df = pd.concat([bloque(port_b, dur_b, fwd_b, bwd_b, fwl_b, bwl_b, syn_b, win_b, ts_b, \"BENIGN\"),\n                    bloque(port_a, dur_a, fwd_a, bwd_a, fwl_a, bwl_a, syn_a, win_a, ts_a, \"DDoS\")], ignore_index=True)\n    df.loc[rng.random(len(df)) < 0.001, \"Flow Bytes/s\"] = np.nan\n    df = pd.concat([df, df.sample(frac=0.01, random_state=seed)], ignore_index=True)\n    return df.sort_values(\" Timestamp\").reset_index(drop=True)\n\n\ndef generar_authlog(ruta=\"auth.log\", n_atacantes=12, seed=42):\n    \"\"\"auth.log sintético de la sesión 2: 40 IP legítimas y n_atacantes IP de fuerza bruta (203.0.113.0/24).\"\"\"\n    import datetime as dt\n    rng = np.random.default_rng(seed); start = dt.datetime(2026, 10, 1)\n    users = [\"root\", \"admin\", \"carlos\", \"ana\", \"luis\", \"deploy\", \"test\", \"oracle\", \"ubuntu\", \"postgres\"]\n    rows = []\n    for i in range(40):\n        ip = f\"10.10.10.{10 + i}\"\n        for t in rng.uniform(0, 7 * 24 * 3600, rng.integers(5, 40)):\n            rows.append((start + dt.timedelta(seconds=float(t)), bool(rng.random() > 0.08), str(rng.choice(users[2:6])), ip))\n    for i in range(n_atacantes):\n        ip = f\"203.0.113.{20 + i}\"; t0 = rng.uniform(0, 7 * 24 * 3600 - 3600)\n        for t in t0 + np.sort(rng.uniform(0, 1800, rng.integers(60, 400))):\n            rows.append((start + dt.timedelta(seconds=float(t)), False, str(rng.choice(users)), ip))\n    rows.sort(key=lambda r: r[0])\n    with open(ruta, \"w\") as f:\n        for ts, ok, user, ip in rows:\n            v = \"Accepted\" if ok else \"Failed\"\n            f.write(f\"{ts:%b %d %H:%M:%S} srv sshd[{rng.integers(1000, 9999)}]: {v} password \"\n                    f\"for {user} from {ip} port {rng.integers(1024, 65000)} ssh2\\n\")\n    return ruta\n\n\ndef auth_features(ruta=\"auth.log\"):\n    \"\"\"Características por IP y ventana de 10 minutos. La columna verdad solo sirve para evaluar.\"\"\"\n    pat = re.compile(r\"^(\\w{3}\\s+\\d+ \\d\\d:\\d\\d:\\d\\d) \\S+ sshd\\[\\d+\\]: (Failed|Accepted) password for (\\S+) from (\\S+) port\")\n    reg = []\n    for linea in open(ruta):\n        m = pat.match(linea)\n        if m:\n            reg.append((pd.to_datetime(\"2026 \" + m.group(1), format=\"%Y %b %d %H:%M:%S\"),\n                        m.group(2) == \"Accepted\", m.group(3), m.group(4)))\n    log = pd.DataFrame(reg, columns=[\"ts\", \"ok\", \"user\", \"ip\"])\n    log[\"win\"] = log[\"ts\"].dt.floor(\"10min\")\n    f = log.groupby([\"ip\", \"win\"]).agg(n=(\"ok\", \"size\"), n_ok=(\"ok\", \"sum\"), n_users=(\"user\", \"nunique\")).reset_index()\n    f[\"n_fail\"] = f[\"n\"] - f[\"n_ok\"]\n    f[\"fail_ratio\"] = f[\"n_fail\"] / f[\"n\"]\n    f[\"verdad\"] = f[\"ip\"].str.startswith(\"203.0.113.\").astype(int)\n    return f\n"
with open("ia_utils.py", "w", encoding="utf-8") as f:
    f.write(IA_UTILS)

import importlib, ia_utils; importlib.reload(ia_utils)
from ia_utils import *
tr, te = load_nsl_kdd("train"), load_nsl_kdd("test")      # descarga NSL-KDD si falta
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn
plt.rcParams["figure.figsize"] = (9, 4)
print("NSL-KDD:", tr.shape, te.shape, "| scikit-learn", sklearn.__version__)

## Parte A · k-means
Cuatro pasos que se repiten: colocar k centros, asignar cada punto al más cercano, mover cada centro a la media de sus puntos, repetir hasta que no se muevan.

In [ ]:
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
tr, te = add_category(tr), add_category(te)
num = ["duration", "src_bytes", "dst_bytes", "count", "srv_count", "serror_rate", "rerror_rate", "same_srv_rate",
       "diff_srv_rate", "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_serror_rate",
       "logged_in", "hot", "num_failed_logins", "num_compromised"]
pre = make_pipeline(FunctionTransformer(np.log1p), StandardScaler()).fit(tr[num])
sub = tr.sample(8000, random_state=0); Xs = pre.transform(sub[num])

def kmeans(X, k, iters=100, seed=0):
    C = X[np.random.default_rng(seed).choice(len(X), k, replace=False)]
    for _ in range(iters):
        d = ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1); lab = d.argmin(1)
        nuevo = np.array([X[lab == j].mean(0) if (lab == j).any() else C[j] for j in range(k)])
        if np.allclose(nuevo, C): break
        C = nuevo
    return lab, C, d.min(1).sum()
lab, C, J = kmeans(Xs, 5)
km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
print("inercia propia", round(J, 1), "| scikit-learn", round(km.inertia_, 1))
print(pd.crosstab(km.labels_, sub["category"]))

In [ ]:
res = []
for k in range(2, 10):
    m = KMeans(n_clusters=k, n_init=5, random_state=0).fit(Xs)
    res.append({"k": k, "inercia": m.inertia_, "silueta": silhouette_score(Xs, m.labels_, sample_size=3000, random_state=0),
                "ARI": adjusted_rand_score(sub["category"], m.labels_)})
r = pd.DataFrame(res).set_index("k"); print(r.round(3))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5)); r.inercia.plot(ax=ax[0], marker="o", title="Codo: inercia")
r.silueta.plot(ax=ax[1], marker="o", title="Silueta (más alto, mejor)"); plt.show()

**Qué deberías ver:** inercias parecidas pero no idénticas (k-means depende del arranque: por eso `n_init`). La tabla cruzada muestra un clúster casi solo de DoS, aunque el algoritmo nunca vio las etiquetas. El codo y la silueta orientan la elección de k.

**Reto avanzado.** Comprueba el ejemplo 1D de la teoría: puntos 1, 2, 10 y 11 con k = 2 dan inercia J = 1. Después quita el escalado y observa cómo `src_bytes` decide solo los grupos.

In [ ]:
# Tu código

## Parte B · DBSCAN: densidad y ruido
No pide k: agrupa zonas densas y marca como **ruido** (−1) lo que queda aislado. `eps` se elige en el codo de la curva de distancias.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
Z = PCA(n_components=8, random_state=0).fit_transform(Xs)
dist, _ = NearestNeighbors(n_neighbors=10).fit(Z).kneighbors(Z)
plt.plot(np.sort(dist[:, -1])); plt.ylim(0, 5); plt.title("Distancia al 10.º vecino, ordenada: eps en el codo"); plt.show()
for eps in [0.6, 1.2, 2.5]:
    db = DBSCAN(eps=eps, min_samples=10).fit(Z); ruido = db.labels_ == -1
    print(f"eps={eps}: clústeres={db.labels_.max() + 1:3d}  ruido={ruido.mean():.3f}  "
          f"ataque en ruido={sub.is_attack[ruido].mean():.2f}  en el resto={sub.is_attack[~ruido].mean():.2f}")

**Reto avanzado.** ¿Por qué reducir con PCA antes de DBSCAN? Relaciónalo con la maldición de la dimensionalidad: en 17 dimensiones todas las distancias se parecen.

In [ ]:
# Tu código

## Parte C · PCA como detector
Entrenamos **solo con tráfico normal**: PCA aprende a resumirlo. Lo que no se parece a lo normal **se reconstruye mal**, y ese error es la puntuación de anomalía.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
normal = tr[tr.is_attack == 0]
pre_n = make_pipeline(FunctionTransformer(np.log1p), StandardScaler()).fit(normal[num])
Xn, Xt = pre_n.transform(normal[num]), pre_n.transform(te[num])
print("varianza acumulada:", np.cumsum(PCA().fit(Xn).explained_variance_ratio_).round(2))
for k in [2, 4, 6, 8, 12]:
    p = PCA(n_components=k).fit(Xn)
    s = ((Xt - p.inverse_transform(p.transform(Xt))) ** 2).sum(1)
    print(f"k={k:2d}  AUC={roc_auc_score(te.is_attack, s):.3f}  AP={average_precision_score(te.is_attack, s):.3f}")

**Qué deberías ver:** con pocos componentes el error separa bien; con demasiados, PCA reconstruye también los ataques y el detector empeora.

**Reto avanzado.** Fija un umbral en el percentil 99 del error sobre el tráfico normal de entrenamiento y calcula recall y tasa de falsas alarmas en el test.

In [ ]:
# Tu código

## Parte D · Isolation Forest, LOF y One-Class SVM

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
import time
Xn_s = Xn[np.random.default_rng(0).choice(len(Xn), 10000, replace=False)]
filas = []
for nom, m in {"Isolation Forest": IsolationForest(n_estimators=300, random_state=0, n_jobs=-1),
               "LOF": LocalOutlierFactor(n_neighbors=35, novelty=True),
               "One-Class SVM": OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")}.items():
    t0 = time.perf_counter(); m.fit(Xn_s); t1 = time.perf_counter()
    s = -m.decision_function(Xt)
    filas.append({"modelo": nom, "AUC": roc_auc_score(te.is_attack, s), "AP": average_precision_score(te.is_attack, s), "t_fit_s": t1 - t0})
    if nom == "Isolation Forest": iso, s_iso = m, s
print(pd.DataFrame(filas).round(3))
te["score_iso"] = s_iso
umbral = np.quantile(-iso.decision_function(Xn_s), 0.99)
print("recall por categoría con el umbral del percentil 99 normal:")
print(te[te.is_attack == 1].groupby("category")["score_iso"].apply(lambda x: (x > umbral).mean()).round(2))

**Qué deberías ver:** AUC entre 0,87 y 0,95 sin ver una sola etiqueta. DoS y probe se detectan mejor que r2l y u2r, que se parecen a una sesión normal.

**Reto avanzado.** Cambia `nu` del One-Class SVM y `contamination` del Isolation Forest y anota el efecto en falsas alarmas. Compara con el Random Forest supervisado de la sesión 5: ¿cuánto se pierde por no tener etiquetas?

In [ ]:
# Tu código

## Parte E · Anomalías en un auth.log, sin etiquetas

In [ ]:
generar_authlog("auth.log", n_atacantes=12)
f = auth_features("auth.log")
cols = ["n", "n_fail", "n_users", "fail_ratio"]
iso_a = IsolationForest(n_estimators=200, contamination=0.05, random_state=0).fit(f[cols])
f["score"] = -iso_a.score_samples(f[cols])
top = f.sort_values("score", ascending=False).head(20)
print(top[["ip", "n", "n_fail", "n_users", "score", "verdad"]].head(10))
print("precision@20:", top["verdad"].mean(), "| AUC:", round(roc_auc_score(f["verdad"], f["score"]), 3))

**Reto avanzado.** Repite con `n_atacantes=2`. ¿Qué ocurre con un administrador legítimo que falla 20 veces su contraseña? Es anomalía y no es malicioso: ¿qué contexto lo desambigua?

In [ ]:
# Tu código

## Parte F · z robusto sobre una serie de DNS
Un día de consultas DNS por minuto con dos horas de exfiltración lenta (el doble de lo habitual). La media y la desviación **se contaminan** con el propio ataque; la mediana y la MAD, no.

In [ ]:
rng = np.random.default_rng(5)
q = rng.poisson(40, 1440).astype(float)
q[800:920] += rng.poisson(40, 120)                 # exfiltración lenta por DNS: 2 horas
z = (q - q.mean()) / q.std()
med = np.median(q); mad = np.median(np.abs(q - med)); zr = 0.6745 * (q - med) / mad
print("minutos marcados · z clásico > 3:", int((np.abs(z) > 3).sum()), "| z robusto > 3,5:", int((np.abs(zr) > 3.5).sum()))
plt.plot(q, lw=0.8); plt.axhline(q.mean() + 3 * q.std(), color="gray", ls="--", label="z clásico = 3")
plt.axhline(med + 3.5 * mad / 0.6745, color="red", label="z robusto = 3,5"); plt.legend(); plt.show()

**Qué deberías ver:** el z clásico marca menos de la mitad de los 120 minutos porque el propio ataque infla la media y la desviación; el z robusto marca casi todos.

**Reto avanzado.** Relaciónalo con un túnel DNS (iodine, dnscat2) y escribe la regla de SIEM equivalente: ventana, agregación y umbral basado en mediana y MAD.

In [ ]:
# Tu código

## Parte G · Presupuesto de alertas
Un SOC no revisa todo: revisa **las k alertas más anómalas**. La métrica realista es **precision@k**.

In [ ]:
orden = te.sort_values("score_iso", ascending=False)
for k in [50, 100, 500, 1000, 5000]:
    print(f"precision@{k:<5d} = {orden.is_attack.head(k).mean():.3f}   ataques capturados: {orden.is_attack.head(k).sum() / te.is_attack.sum():.3f}")

**Reto avanzado.** Simula el bucle con el analista: de las 100 primeras alertas, toma sus etiquetas verdaderas, entrena un `RandomForestClassifier` pequeño solo con ellas y reordena el resto. ¿Mejora precision@100 en la siguiente tanda? Escribe media página sobre cobertura frente a fatiga de alertas.

In [ ]:
# Tu código

## Entrega
Reinicia y ejecuta todo. Entrega `ia6_anomalias.ipynb` con la tabla de detectores (AUC, AP, tiempo, necesita escalar, entrena solo con normal) y un texto breve, «Anomalía no es malicioso», con tres falsos positivos legítimos y cómo los desambiguarías.